# Phase 8: Theory and Sensitivity Analysis

This notebook completes four tasks:
1. Theoretical validation (monotonicity and tractability)
2. Sensitivity on budget frontier (Model 1 vs Model 2)
3. Sensitivity on efficiency-equity trade-off (Model 1)
4. Sensitivity on weather penalties (Model 2, Heavy Rain mild vs severe)

All experiments reuse the processed artifacts from Phases 3-7 and keep the optimization models linear.

## 8.1 Theoretical Validation

### Monotonicity of Model 2 (Robust Max-Min) with respect to budget $B$

Let the feasible set under budget $B$ be
$$
\mathcal{X}(B)=\left\{x\in\{0,1\}^{|J|}:\sum_{j\in J} c_j x_j \le B\right\}.
$$
If $B_1 \le B_2$, then every $x\in\mathcal{X}(B_1)$ also satisfies the looser budget, so
$$
\mathcal{X}(B_1)\subseteq \mathcal{X}(B_2).
$$
Model 2 maximizes
$$
Z(x)=\min_{s\in S}\left(\bar C^{\mathrm{vul}}_s + \sum_{j\in J}\Delta^{\mathrm{vul}}_{js}x_j\right).
$$
Because the objective function is the same while the feasible set expands with larger $B$, the optimal value is monotone non-decreasing:
$$
\operatorname{OPT}(B_1)\le \operatorname{OPT}(B_2),\quad B_1\le B_2.
$$

### Tractability under linear additive marginal gains

Assume additive gains $\Delta_{gjs}$ are precomputed constants and decisions are binary $x_j\in\{0,1\}$.

For Model 1, the objective is a linear form:
$$
\max_x\;\sum_{j\in J} a_j x_j - \lambda\sum_{j\in J} b_j x_j
$$
with one linear budget constraint and binary variables, i.e., a standard 0-1 ILP.

For Model 2, introduce auxiliary scalar $Z$ with constraints
$$
Z \le \bar C^{\mathrm{vul}}_s + \sum_{j\in J}\Delta^{\mathrm{vul}}_{js}x_j,\;\forall s\in S,
$$
and maximize $Z$ under the same budget and binary constraints. This is a standard MILP (still linear in all variables).

Hence both models are solvable by branch-and-bound / branch-and-cut MILP solvers.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import importlib
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

opt_solver = importlib.import_module("src.optimization.solver")

processed_dir = repo_root / "data" / "processed"
solver_script = repo_root / "src" / "optimization" / "solver.py"

candidate_path = processed_dir / "candidate_library.csv"
delta_path = processed_dir / "marginal_gains_delta.csv"
baseline_path = processed_dir / "baseline_connectivity_scores.csv"
weather_prob_path = processed_dir / "weather_probabilities.csv"
demographics_path = processed_dir / "acs_demographics_centroids.geojson"

print("repo_root:", repo_root)
print("solver_script exists:", solver_script.exists())

In [ ]:
phase8_eff_frontier = processed_dir / "phase8_efficiency_frontier.csv"
phase8_robust_frontier = processed_dir / "phase8_robust_frontier.csv"
phase8_eff_fairness = processed_dir / "phase8_efficiency_fairness.csv"
phase8_robust_fairness = processed_dir / "phase8_robust_fairness.csv"


def run_solver_sweep(
    budgets: str,
    lambdas: str,
    eff_out: Path,
    robust_out: Path,
    probability_profile: str = "Annual",
    max_seconds: int = 45,
    force_rerun: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if eff_out.exists() and robust_out.exists() and not force_rerun:
        return pd.read_csv(eff_out), pd.read_csv(robust_out)

    cmd = [
        sys.executable,
        str(solver_script),
        "--probability-profile",
        probability_profile,
        "--budgets",
        budgets,
        "--lambdas",
        lambdas,
        "--max-seconds",
        str(max_seconds),
        "--solver",
        "cbc",
        "--efficiency-output",
        str(eff_out),
        "--robust-output",
        str(robust_out),
    ]

    run = subprocess.run(
        cmd,
        cwd=repo_root,
        text=True,
        capture_output=True,
        check=True,
    )

    print(run.stdout)
    if run.stderr.strip():
        print("stderr:")
        print(run.stderr)

    return pd.read_csv(eff_out), pd.read_csv(robust_out)


def parse_selected_list(text: object) -> list[str]:
    if pd.isna(text):
        return []
    return [token for token in str(text).split("|") if token]


def monotonicity_violations(df: pd.DataFrame, value_col: str = "objective_value", tol: float = 1e-6) -> int:
    ordered = df.sort_values("budget")[value_col].to_numpy(dtype=float)
    diffs = np.diff(ordered)
    return int(np.sum(diffs < -tol))

## 8.2 Sensitivity: Budget Frontier

We run a budget sweep and compare both models on objective value versus budget, then inspect marginal objective gain per unit budget to diagnose diminishing returns.

In [ ]:
frontier_budgets = "500,1000,1500,2000,3000,4000,5000"
frontier_lambdas = "0"

fairness_budget = "2000"
fairness_lambdas = "0,0.25,0.5,1,2,3,5"

eff_frontier_df, robust_frontier_df = run_solver_sweep(
    budgets=frontier_budgets,
    lambdas=frontier_lambdas,
    eff_out=phase8_eff_frontier,
    robust_out=phase8_robust_frontier,
    max_seconds=45,
    force_rerun=False,
)

eff_fairness_df, robust_fairness_df = run_solver_sweep(
    budgets=fairness_budget,
    lambdas=fairness_lambdas,
    eff_out=phase8_eff_fairness,
    robust_out=phase8_robust_fairness,
    max_seconds=45,
    force_rerun=False,
)

print("Frontier rows (Model 1):", len(eff_frontier_df))
print("Frontier rows (Model 2):", len(robust_frontier_df))
print("Fairness rows (Model 1):", len(eff_fairness_df))
print("Fairness rows (Model 2):", len(robust_fairness_df))

In [ ]:
frontier_model1 = eff_frontier_df.copy()
frontier_model1["lambda"] = pd.to_numeric(frontier_model1["lambda"], errors="coerce")
frontier_model1 = frontier_model1[np.isclose(frontier_model1["lambda"].fillna(-1), 0.0)]
frontier_model1 = frontier_model1.sort_values("budget")

frontier_model2 = robust_frontier_df.sort_values("budget").copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

axes[0].plot(
    frontier_model1["budget"],
    frontier_model1["objective_value"],
    marker="o",
    linewidth=2,
    label="Model 1 objective (lambda=0)",
)
axes[0].plot(
    frontier_model2["budget"],
    frontier_model2["objective_value"],
    marker="s",
    linewidth=2,
    label="Model 2 robust floor objective",
)
axes[0].set_title("Budget vs Objective Value")
axes[0].set_xlabel("Budget B")
axes[0].set_ylabel("Objective Value")
axes[0].legend(loc="best")

m1 = frontier_model1[["budget", "objective_value"]].copy().sort_values("budget")
m1["delta_obj"] = m1["objective_value"].diff()
m1["delta_budget"] = m1["budget"].diff()
m1["marginal_gain_per_budget"] = m1["delta_obj"] / m1["delta_budget"]

m2 = frontier_model2[["budget", "objective_value"]].copy().sort_values("budget")
m2["delta_obj"] = m2["objective_value"].diff()
m2["delta_budget"] = m2["budget"].diff()
m2["marginal_gain_per_budget"] = m2["delta_obj"] / m2["delta_budget"]

axes[1].plot(
    m1["budget"],
    m1["marginal_gain_per_budget"],
    marker="o",
    linewidth=2,
    label="Model 1 marginal gain",
)
axes[1].plot(
    m2["budget"],
    m2["marginal_gain_per_budget"],
    marker="s",
    linewidth=2,
    label="Model 2 marginal gain",
)
axes[1].set_title("Diminishing Returns Diagnostic")
axes[1].set_xlabel("Budget B")
axes[1].set_ylabel("Delta Objective / Delta Budget")
axes[1].legend(loc="best")

plt.show()

print("Model 2 monotonicity violations:", monotonicity_violations(frontier_model2))
print("Model 1 monotonicity violations:", monotonicity_violations(frontier_model1))

m1[["budget", "objective_value", "marginal_gain_per_budget"]]

**Observation (Budget Frontier):** if the second panel slopes downward as budget grows, the model is exhibiting diminishing returns. This is expected because high bang-for-buck candidates are selected first, then remaining options are progressively less efficient.

## 8.3 Sensitivity: Fairness Trade-off

At fixed budget, we vary $\lambda$ and plot fairness gap (x-axis) against total population-weighted connectivity (y-axis).

In [ ]:
inputs, input_info = opt_solver.build_optimization_inputs(
    candidate_library_path=candidate_path,
    delta_path=delta_path,
    baseline_path=baseline_path,
    demographics_path=demographics_path,
    weather_probability_path=weather_prob_path,
    probability_profile="Annual",
    acs_population_path=opt_solver.detect_latest_acs_tract_file(repo_root / "data" / "raw" / "acs"),
)
print(input_info)

baseline_df = pd.read_csv(baseline_path)
baseline_df["tract_id"] = baseline_df["tract_id"].map(opt_solver.normalize_tract_id)
baseline_df["weather_scenario"] = baseline_df["weather_scenario"].astype(str).str.strip()
baseline_df["connectivity_score"] = pd.to_numeric(baseline_df["connectivity_score"], errors="coerce").fillna(0.0)

baseline_df["alpha_weight"] = baseline_df["tract_id"].map(inputs.alpha_weights).fillna(0.0)
baseline_df["scenario_probability"] = baseline_df["weather_scenario"].map(inputs.scenario_probabilities).fillna(0.0)

baseline_population_weighted_connectivity = float(
    (
        baseline_df["connectivity_score"]
        * baseline_df["alpha_weight"]
        * baseline_df["scenario_probability"]
    ).sum()
)


def selected_efficiency_gain(selected_text: object) -> float:
    selected = parse_selected_list(selected_text)
    return float(sum(inputs.efficiency_coeff.get(candidate, 0.0) for candidate in selected))


tradeoff_df = eff_fairness_df.copy()
tradeoff_df["lambda"] = pd.to_numeric(tradeoff_df["lambda"], errors="coerce")
tradeoff_df = tradeoff_df[tradeoff_df["status"].astype(str).str.contains("OPTIMAL", na=False)].copy()
tradeoff_df["efficiency_gain"] = tradeoff_df["selected_candidates_list"].apply(selected_efficiency_gain)
tradeoff_df["total_population_weighted_connectivity"] = (
    baseline_population_weighted_connectivity + tradeoff_df["efficiency_gain"]
)
tradeoff_df = tradeoff_df.sort_values("lambda")

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    tradeoff_df["gap"],
    tradeoff_df["total_population_weighted_connectivity"],
    c=tradeoff_df["lambda"],
    cmap="viridis",
    s=140,
)
ax.plot(
    tradeoff_df["gap"],
    tradeoff_df["total_population_weighted_connectivity"],
    linewidth=1.5,
    alpha=0.7,
)
for _, row in tradeoff_df.iterrows():
    ax.annotate(
        f"lambda={row['lambda']:g}",
        (row["gap"], row["total_population_weighted_connectivity"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
    )

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("lambda")
ax.set_title("Model 1: Efficiency vs Equity Trade-off")
ax.set_xlabel("Fairness Gap (non-vulnerable - vulnerable)")
ax.set_ylabel("Total Population-Weighted Connectivity")
plt.show()

tradeoff_df[["budget", "lambda", "gap", "total_population_weighted_connectivity", "objective_value"]]

**Observation (Fairness Trade-off):** as $\lambda$ increases, solutions typically move toward smaller fairness gaps, with some sacrifice in total population-weighted connectivity. The plotted trajectory is the empirical efficiency-equity frontier under fixed budget.

## 8.4 Sensitivity: Weather Penalties

We perturb Heavy Rain penalties and rerun Model 2 at fixed budget to compare selected candidate lists and resilience-oriented candidate characteristics.

In [ ]:
runtime_df = pd.read_csv(candidate_path, usecols=["candidate_id", "avg_trip_runtime_h"])
runtime_df["avg_trip_runtime_h"] = pd.to_numeric(runtime_df["avg_trip_runtime_h"], errors="coerce")
runtime_df["avg_trip_runtime_h"] = runtime_df["avg_trip_runtime_h"].fillna(runtime_df["avg_trip_runtime_h"].median())

r_min = float(runtime_df["avg_trip_runtime_h"].min())
r_max = float(runtime_df["avg_trip_runtime_h"].max())
den = max(r_max - r_min, 1e-9)
runtime_df["runtime_norm"] = (runtime_df["avg_trip_runtime_h"] - r_min) / den
runtime_norm_map = runtime_df.set_index("candidate_id")["runtime_norm"].to_dict()


def apply_heavy_rain_penalty(
    inputs_obj: opt_solver.OptimizationInputs,
    severity_strength: float,
    baseline_penalty: float,
):
    heavy_scenarios = [s for s in inputs_obj.scenarios if "heavy rain" in s.lower()]
    if not heavy_scenarios:
        raise ValueError("No 'Heavy Rain' scenario found in inputs.scenarios")

    updated_vuln_scenario_coeff = {}
    for scenario in inputs_obj.scenarios:
        raw_map = inputs_obj.vulnerable_scenario_coeff.get(scenario, {})
        if scenario in heavy_scenarios:
            adjusted_map = {}
            for candidate, value in raw_map.items():
                runtime_norm = float(runtime_norm_map.get(candidate, 0.5))
                candidate_factor = max(0.05, 1.0 - severity_strength * runtime_norm)
                adjusted_map[candidate] = float(value) * candidate_factor
            updated_vuln_scenario_coeff[scenario] = adjusted_map
        else:
            updated_vuln_scenario_coeff[scenario] = {
                candidate: float(value)
                for candidate, value in raw_map.items()
            }

    updated_baseline_vuln = {
        scenario: float(inputs_obj.baseline_vulnerable_scenario_avg.get(scenario, 0.0))
        * (baseline_penalty if scenario in heavy_scenarios else 1.0)
        for scenario in inputs_obj.scenarios
    }

    return (
        replace(
            inputs_obj,
            vulnerable_scenario_coeff=updated_vuln_scenario_coeff,
            baseline_vulnerable_scenario_avg=updated_baseline_vuln,
        ),
        heavy_scenarios,
    )


solver_name, solver_label = opt_solver.choose_solver_name("cbc")
analysis_budget = 1000.0

mild_inputs, heavy_scenarios = apply_heavy_rain_penalty(
    inputs,
    severity_strength=0.25,
    baseline_penalty=0.90,
)
severe_inputs, _ = apply_heavy_rain_penalty(
    inputs,
    severity_strength=0.75,
    baseline_penalty=0.70,
)

mild_row = opt_solver.solve_robust_max_min(
    inputs=mild_inputs,
    budget=analysis_budget,
    solver_name=solver_name,
    solver_label=solver_label,
    max_seconds=60,
    threads=0,
    seed=42,
)
severe_row = opt_solver.solve_robust_max_min(
    inputs=severe_inputs,
    budget=analysis_budget,
    solver_name=solver_name,
    solver_label=solver_label,
    max_seconds=60,
    threads=0,
    seed=42,
)

mild_set = set(parse_selected_list(mild_row["selected_candidates_list"]))
severe_set = set(parse_selected_list(severe_row["selected_candidates_list"]))
common_set = mild_set & severe_set
union_set = mild_set | severe_set
mild_only = sorted(mild_set - severe_set)
severe_only = sorted(severe_set - mild_set)

jaccard = (len(common_set) / len(union_set)) if union_set else np.nan

selection_compare = pd.DataFrame(
    [
        {
            "config": "Mild Heavy Rain penalty (runtime-weighted)",
            "objective_value": mild_row["objective_value"],
            "robust_floor": mild_row["robust_floor"],
            "selected_count": len(mild_set),
            "used_budget": mild_row["used_budget"],
        },
        {
            "config": "Severe Heavy Rain penalty (runtime-weighted)",
            "objective_value": severe_row["objective_value"],
            "robust_floor": severe_row["robust_floor"],
            "selected_count": len(severe_set),
            "used_budget": severe_row["used_budget"],
        },
    ]
)

print("Heavy rain scenarios:", heavy_scenarios)
print("Analysis budget:", analysis_budget)
print("Jaccard similarity:", round(float(jaccard), 4))
print("Mild-only candidates:", len(mild_only))
print("Severe-only candidates:", len(severe_only))
selection_compare

In [ ]:
candidate_cols = [
    "candidate_id",
    "route_id",
    "direction_id",
    "time_period",
    "cost_cj",
    "avg_trip_runtime_h",
    "service_gain_scalar",
    "headway_improvement_mins",
]
candidate_df = pd.read_csv(candidate_path, usecols=candidate_cols)
for col in ["cost_cj", "avg_trip_runtime_h", "service_gain_scalar", "headway_improvement_mins"]:
    candidate_df[col] = pd.to_numeric(candidate_df[col], errors="coerce")

delta_df = pd.read_csv(delta_path)
delta_df["delta_score"] = pd.to_numeric(delta_df["delta_score"], errors="coerce").fillna(0.0)
heavy_mask = delta_df["weather_s"].astype(str).str.contains("Heavy Rain", case=False, na=False)
heavy_gain = delta_df.loc[heavy_mask].groupby("candidate_j")["delta_score"].sum()

candidate_df["heavy_gain"] = candidate_df["candidate_id"].map(heavy_gain).fillna(0.0)
candidate_df["heavy_gain_per_cost"] = candidate_df["heavy_gain"] / candidate_df["cost_cj"].replace(0, np.nan)


def profile_selected(selected_ids: set[str], label: str) -> dict[str, float | str | int]:
    subset = candidate_df[candidate_df["candidate_id"].isin(selected_ids)].copy()
    return {
        "group": label,
        "count": int(len(subset)),
        "avg_trip_runtime_h": float(subset["avg_trip_runtime_h"].mean()),
        "avg_service_gain_scalar": float(subset["service_gain_scalar"].mean()),
        "avg_headway_improvement_mins": float(subset["headway_improvement_mins"].mean()),
        "avg_heavy_gain_per_cost": float(subset["heavy_gain_per_cost"].mean()),
    }


profile_df = pd.DataFrame(
    [
        profile_selected(mild_set, "Mild config selected"),
        profile_selected(severe_set, "Severe config selected"),
        profile_selected(set(mild_only), "Mild-only"),
        profile_selected(set(severe_only), "Severe-only"),
    ]
)

plot_df = profile_df[profile_df["group"].isin(["Mild config selected", "Severe config selected"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sns.barplot(
    data=plot_df,
    x="group",
    y="avg_trip_runtime_h",
    hue="group",
    legend=False,
    ax=axes[0],
    palette="Set2",
)
axes[0].set_title("Selected Candidate Average Runtime")
axes[0].set_xlabel("")
axes[0].set_ylabel("Avg trip runtime (hours)")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(
    data=plot_df,
    x="group",
    y="avg_heavy_gain_per_cost",
    hue="group",
    legend=False,
    ax=axes[1],
    palette="Set2",
)
axes[1].set_title("Selected Candidate Heavy-Rain Gain per Cost")
axes[1].set_xlabel("")
axes[1].set_ylabel("Avg heavy-rain gain / cost")
axes[1].tick_params(axis="x", rotation=15)

plt.show()

severe_only_table = (
    candidate_df[candidate_df["candidate_id"].isin(severe_only)]
    .sort_values("heavy_gain_per_cost", ascending=False)
    .head(20)
)

profile_df, severe_only_table

**Observation (Weather Penalty Sensitivity):** compare the mild vs severe selection sets using overlap and severe-only table.

Practical interpretation rule used here:
- Lower `avg_trip_runtime_h` is treated as a short-distance service proxy.
- Higher `heavy_gain_per_cost` indicates better resilience payoff under Heavy Rain.

If severe configuration shifts toward lower runtime and/or higher heavy-rain gain-per-cost, that supports the claim that severe weather pressure favors more weather-resilient, often shorter and more direct restoration options.

Note: this dataset does not carry an explicit transfer-count field at candidate level, so "zero-transfer" is evaluated indirectly via runtime and heavy-rain gain proxies.

In [ ]:
phase8_weather_compare_path = processed_dir / "phase8_weather_penalty_compare.csv"
phase8_weather_profile_path = processed_dir / "phase8_weather_penalty_profiles.csv"
phase8_tradeoff_path = processed_dir / "phase8_fairness_tradeoff.csv"

selection_compare.to_csv(phase8_weather_compare_path, index=False)
profile_df.to_csv(phase8_weather_profile_path, index=False)
tradeoff_df.to_csv(phase8_tradeoff_path, index=False)

print("Saved:")
print(" -", phase8_weather_compare_path)
print(" -", phase8_weather_profile_path)
print(" -", phase8_tradeoff_path)